# BAC Datathon — Round 2 starter kit

**How to use on the night:** fill in the **CONFIG** cell (file name + target column), then *Run All*.
Everything below it adapts to whatever dataset they hand you.

| Section | What it gives you | Rough time |
|---|---|---|
| 1. Load + profile | What's in the data, what's broken | 5 min |
| 2. Clean | Dates parsed, `$1,200` → `1200`, duplicates gone | 5 min |
| 3. Business slicing | `by()`, `trend()`, `top()` — the numbers that go on slides | 30–45 min |
| 4. Auto EDA vs target | Which columns move the outcome | 10 min |
| 5. Model | Baselines vs real models, one clean metrics table | 15 min |
| 6. Drivers | Which features matter most (in plain column names) | 5 min |
| 7. Lift / impact | "Target the top 20% and you catch X% of churners" | 5 min |
| 8. Segments | Customer groups with a profile each | optional |

Charts save to `charts/` as PNGs, ready to drop into slides.

In [ ]:
# 0. SETUP CHECK — run this on Thursday night, not Friday
import sys, os, re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
print("python ", sys.version.split()[0])
print("pandas ", pd.__version__, "| numpy", np.__version__, "| sklearn", sklearn.__version__, "| matplotlib", matplotlib.__version__)
try:
    import openpyxl; print("openpyxl", openpyxl.__version__, "(needed for .xlsx)")
except ImportError:
    print("!! openpyxl missing -> pip install openpyxl   (needed to read Excel files)")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
os.makedirs("charts", exist_ok=True)

# Slide-friendly chart style: big fonts, no chart junk
plt.rcParams.update({
    "figure.figsize": (9, 5), "figure.dpi": 110, "savefig.dpi": 200,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 15, "axes.titleweight": "bold", "axes.labelsize": 12,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.frameon": False,
    "axes.prop_cycle": matplotlib.cycler(color=["#2563eb", "#f97316", "#10b981", "#8b5cf6", "#ef4444", "#64748b"]),
})
ACCENT, GREY = "#2563eb", "#cbd5e1"

from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype, is_object_dtype, is_string_dtype, is_bool_dtype
def is_text(s):
    # works on pandas 2.x (object) and 3.x (str dtype)
    return (is_object_dtype(s) or is_string_dtype(s)) and not is_numeric_dtype(s)
def text_cols(d):
    return [c for c in d.columns if is_text(d[c]) or isinstance(d[c].dtype, pd.CategoricalDtype) or is_bool_dtype(d[c])]

def save_chart(name, fig=None):
    # Saves the current chart to charts/<name>.png for the deck
    fig = fig or plt.gcf()
    fig.tight_layout()
    path = os.path.join("charts", re.sub(r"[^a-z0-9_]+", "_", name.lower()) + ".png")
    fig.savefig(path, bbox_inches="tight")
    print("saved ->", path)

## CONFIG — the only cell you *have* to edit

In [ ]:
DATA_PATH  = "data.csv"   # .csv / .xlsx / .xls / .json / .parquet
SHEET      = 0            # Excel only: sheet name or index. Set None to load ALL sheets into a dict
TARGET     = None         # column you want to predict/explain, e.g. "churn". None = skip modelling
ID_COLS    = []           # columns that are just IDs (customer_id etc.) — excluded from modelling
DROP_COLS  = []           # anything that leaks the answer (e.g. "cancellation_date" when predicting churn)
TEST_SIZE  = 0.2
SEED       = 42

## 1. Load + profile

In [ ]:
def snake(c):
    c = re.sub(r"[^0-9a-zA-Z]+", "_", str(c).strip()).strip("_").lower()
    return c or "col"

def load(path, sheet=0):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".xlsx", ".xls", ".xlsm"):
        out = pd.read_excel(path, sheet_name=sheet)
    elif ext == ".json":
        out = pd.read_json(path)
    elif ext == ".parquet":
        out = pd.read_parquet(path)
    else:
        try:
            out = pd.read_csv(path)
        except UnicodeDecodeError:
            out = pd.read_csv(path, encoding="latin-1")
    fix = lambda d: d.rename(columns=snake)
    return {k: fix(v) for k, v in out.items()} if isinstance(out, dict) else fix(out)

raw = load(DATA_PATH, SHEET)
if isinstance(raw, dict):
    for name, d in raw.items():
        print(f"sheet '{name}': {d.shape}  cols: {list(d.columns)[:12]}")
    print("\nMultiple sheets loaded. Pick/merge one, e.g.  df = raw['orders'].merge(raw['customers'], on='customer_id', how='left')")
    df = next(iter(raw.values())).copy()
else:
    df = raw.copy()

TARGET  = snake(TARGET) if TARGET else None
ID_COLS = [snake(c) for c in ID_COLS]
DROP_COLS = [snake(c) for c in DROP_COLS]
print("shape:", df.shape)
df.head()

In [ ]:
def profile(d):
    # One row per column: type, % missing, # unique, an example value
    p = pd.DataFrame({
        "dtype": d.dtypes.astype(str),
        "missing_%": (d.isna().mean() * 100).round(1),
        "n_unique": d.nunique(),
        "example": [d[c].dropna().iloc[0] if d[c].notna().any() else None for c in d.columns],
    })
    return p.sort_values("missing_%", ascending=False)

print(f"{len(df):,} rows x {df.shape[1]} cols | exact duplicate rows: {df.duplicated().sum():,}")
profile(df)

In [ ]:
df.describe(include="all").T

## 2. Clean (auto) — read the printout, it tells you what changed

In [ ]:
def auto_clean(d, verbose=True):
    d = d.copy()
    log = []
    n0 = len(d)
    d = d.drop_duplicates()
    if len(d) < n0: log.append(f"dropped {n0-len(d):,} duplicate rows")
    for c in d.columns:
        s = d[c]
        if is_text(s):
            s = s.astype(object)
            s2 = s.astype(str).str.strip()
            d[c] = s.where(s.isna(), s2).replace({"": np.nan, "nan": np.nan, "NaN": np.nan, "None": np.nan, "NA": np.nan, "N/A": np.nan, "-": np.nan})
            s = d[c]
            # "$1,200" / "45%" / "1,000" -> numbers
            stripped = s.dropna().astype(str).str.replace(r"[$,%\s]|SGD|S\$", "", regex=True)
            num = pd.to_numeric(stripped, errors="coerce")
            if len(stripped) and num.notna().mean() > 0.95:
                d[c] = pd.to_numeric(s.astype(str).str.replace(r"[$,%\s]|SGD|S\$", "", regex=True), errors="coerce")
                log.append(f"{c}: text -> number"); continue
            # dates
            if re.search(r"date|time|_at$|_on$|month|day|year|period", c) or s.dropna().astype(str).str.match(r"^\d{4}-\d{2}-\d{2}|^\d{1,2}/\d{1,2}/\d{2,4}").mean() > 0.9:
                dt = pd.to_datetime(s, errors="coerce", dayfirst=False)
                if dt.notna().mean() > 0.9 * s.notna().mean():
                    d[c] = dt; log.append(f"{c}: text -> datetime"); continue
            # Yes/No style columns -> 1/0
            low = s.dropna().astype(str).str.lower()
            if low.nunique() == 2 and set(low.unique()) <= {"yes","no","y","n","true","false","t","f","1","0"}:
                d[c] = s.astype(str).str.lower().map({"yes":1,"y":1,"true":1,"t":1,"1":1,"no":0,"n":0,"false":0,"f":0,"0":0})
                log.append(f"{c}: yes/no -> 1/0")
    if verbose:
        print("\n".join(log) if log else "nothing needed fixing")
    return d

df = auto_clean(df)

# Add calendar parts for every datetime column (useful for trends / seasonality)
DATE_COLS = [c for c in df.columns if is_datetime64_any_dtype(df[c])]
for c in DATE_COLS:
    df[f"{c}_year"] = df[c].dt.year
    df[f"{c}_month"] = df[c].dt.month
    df[f"{c}_dow"] = df[c].dt.dayofweek   # 0 = Monday
print("date columns:", DATE_COLS)
profile(df).head(15)

## 3. Business slicing — this is where most of the pitch comes from

Judges remember **one number per slide**. These helpers get you those numbers fast:

* `by(df, "region", "revenue")` → total, mean, count and **share of total** per group
* `rate(df, "plan_type", "churn")` → churn rate per group vs overall
* `trend(df, "order_date", "revenue", "M")` → monthly line chart
* `top(df, "product", "revenue", n=10)` → top-N bar chart

In [ ]:
def by(d, group, metric=None, agg="sum", sort=True):
    # Group summary with each group's share of the total
    if metric is None:
        out = d[group].value_counts(dropna=False).to_frame("count")
        out["share_%"] = out["count"] / out["count"].sum() * 100
        return out
    g = d.groupby(group, dropna=False)[metric]
    out = pd.DataFrame({"total": g.sum(), "mean": g.mean(), "median": g.median(), "count": g.size()})
    out["share_of_total_%"] = out["total"] / out["total"].sum() * 100
    return out.sort_values(agg if agg in out else "total", ascending=False) if sort else out

def rate(d, group, flag, min_n=20, plot=True, name=None):
    # Rate of a 0/1 outcome per group, vs the overall rate (lift)
    overall = d[flag].mean()
    g = d.groupby(group, dropna=False)[flag].agg(rate="mean", n="size")
    g = g[g["n"] >= min_n].sort_values("rate", ascending=False)
    g["lift_vs_avg"] = g["rate"] / overall
    if plot and len(g):
        ax = (g["rate"] * 100).head(15).iloc[::-1].plot.barh(color=[ACCENT if r > overall else GREY for r in g["rate"].head(15).iloc[::-1]])
        ax.axvline(overall * 100, ls="--", c="black", lw=1); ax.text(overall * 100, ax.get_ylim()[1], f" avg {overall:.1%}", fontsize=10, va="bottom")
        ax.set_xlabel(f"{flag} rate (%)"); ax.set_ylabel(""); ax.set_title(f"{flag} rate by {group}")
        save_chart(name or f"rate_{flag}_by_{group}"); plt.show()
    return g

def trend(d, date_col, metric=None, freq="M", agg="sum", name=None):
    # Line chart over time. freq: D / W / M / Q / Y
    s = d.set_index(date_col).sort_index()
    try:                       # pandas < 2.2 wants "M"; pandas >= 3 wants "ME"
        r = (s[metric] if metric else s).resample(freq)
    except ValueError:
        r = (s[metric] if metric else s).resample(freq + "E")
    s = r.agg(agg) if metric else r.size()
    ax = s.plot(marker="o", lw=2, color=ACCENT)
    ax.set_title(f"{metric or 'count'} ({agg}) per {freq}"); ax.set_xlabel("")
    save_chart(name or f"trend_{metric or 'count'}"); plt.show()
    change = (s.iloc[-1] / s.iloc[0] - 1) * 100 if len(s) > 1 and s.iloc[0] else np.nan
    print(f"first {s.iloc[0]:,.2f} -> last {s.iloc[-1]:,.2f} ({change:+.1f}%) | peak {s.idxmax():%Y-%m-%d} at {s.max():,.2f}")
    return s

def top(d, group, metric=None, n=10, agg="sum", name=None):
    # Horizontal bar of the top-N groups; prints how concentrated it is
    s = d.groupby(group)[metric].agg(agg) if metric else d[group].value_counts()
    s = s.sort_values(ascending=False)
    head = s.head(n)
    ax = head.iloc[::-1].plot.barh(color=ACCENT)
    ax.set_title(f"Top {n} {group} by {metric or 'count'}"); ax.set_ylabel("")
    save_chart(name or f"top_{group}"); plt.show()
    if agg == "sum" or metric is None:
        print(f"top {n} of {len(s)} {group}s = {head.sum() / s.sum():.1%} of total")
    return head

# Examples — uncomment and swap in real column names:
# by(df, "region", "revenue")
# rate(df, "plan_type", TARGET)
# trend(df, DATE_COLS[0], "revenue", "M")
# top(df, "product", "revenue")

## 4. Auto EDA against the target

In [ ]:
num_cols = [c for c in df.select_dtypes(include="number").columns if c not in ID_COLS + DROP_COLS]
cat_cols = [c for c in text_cols(df) if c not in ID_COLS + DROP_COLS]

# Distributions of numeric columns (spot skew + outliers)
show = [c for c in num_cols if c != TARGET][:12]
if show:
    ncol = 4; nrow = int(np.ceil(len(show) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(16, 3.2 * nrow)); axes = np.array(axes).ravel()
    for ax, c in zip(axes, show):
        df[c].dropna().hist(ax=ax, bins=30, color=ACCENT, alpha=.85); ax.set_title(c, fontsize=11)
        ax.grid(False)
    for ax in axes[len(show):]: ax.axis("off")
    plt.suptitle("Numeric distributions", fontweight="bold"); save_chart("distributions"); plt.show()

if TARGET and TARGET in df:
    y_all = df[TARGET]
    IS_CLF = is_text(y_all) or (y_all.nunique() <= 10)
    print("task:", "CLASSIFICATION" if IS_CLF else "REGRESSION")
    print(y_all.value_counts(normalize=True).head(10) if IS_CLF else y_all.describe())

    # Correlation of numeric features with the target (binary or numeric)
    y_num = y_all if (not IS_CLF) or (y_all.nunique() == 2 and is_numeric_dtype(y_all)) else None
    if y_num is not None:
        corr = df[[c for c in num_cols if c != TARGET]].corrwith(y_num).dropna().sort_values(key=abs, ascending=False)
        ax = corr.head(12).iloc[::-1].plot.barh(color=[ACCENT if v > 0 else "#f97316" for v in corr.head(12).iloc[::-1]])
        ax.set_title(f"Correlation with {TARGET}"); ax.axvline(0, c="black", lw=.8)
        save_chart("corr_with_target"); plt.show()
        display(corr.head(12).to_frame("corr"))

    # Categorical columns: outcome rate per category (binary) or mean (regression)
    for c in [c for c in cat_cols if 2 <= df[c].nunique() <= 25][:6]:
        if IS_CLF and y_num is not None:
            display(rate(df, c, TARGET, min_n=10))
        elif not IS_CLF:
            display(by(df, c, TARGET).sort_values("mean", ascending=False).head(10))
else:
    print("No TARGET set — skipping target analysis. Use section 3 helpers to explore.")

## 5. Model — baselines first, then real models

Always show the **dummy baseline**. "Our model beats guessing-the-majority by X points" is a sentence judges understand.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn import metrics as M

def make_ohe():
    try:    return OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=10)
    except TypeError:  # older sklearn
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

results, fitted = None, {}
if TARGET and TARGET in df:
    data = df.dropna(subset=[TARGET]).copy()
    y = data[TARGET]
    if IS_CLF and not is_numeric_dtype(y):
        y = y.astype("category").cat.codes  # text labels -> 0/1/2...
    excl = set(ID_COLS + DROP_COLS + [TARGET] + DATE_COLS)
    feats_num = [c for c in data.select_dtypes(include="number").columns if c not in excl]
    feats_cat = [c for c in text_cols(data)
                 if c not in excl and data[c].nunique() <= 50]   # skip free-text / high-cardinality
    X = data[feats_num + feats_cat]
    print(f"features: {len(feats_num)} numeric, {len(feats_cat)} categorical | rows: {len(X):,}")

    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), feats_num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]), feats_cat),
    ])
    strat = y if IS_CLF and y.value_counts().min() >= 2 else None
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=strat)

    if IS_CLF:
        models = {
            "Baseline (always majority)": DummyClassifier(strategy="most_frequent"),
            "Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
            "Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced", n_jobs=-1, random_state=SEED),
            "Gradient boosting": HistGradientBoostingClassifier(random_state=SEED),
        }
    else:
        models = {
            "Baseline (always mean)": DummyRegressor(),
            "Ridge regression": Ridge(),
            "Random forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=SEED),
            "Gradient boosting": HistGradientBoostingRegressor(random_state=SEED),
        }

    rows = []
    binary = IS_CLF and y.nunique() == 2
    for name, m in models.items():
        pipe = Pipeline([("pre", pre), ("model", m)]).fit(X_tr, y_tr)
        fitted[name] = pipe
        p = pipe.predict(X_te)
        if IS_CLF:
            r = {"model": name, "accuracy": M.accuracy_score(y_te, p),
                 "precision": M.precision_score(y_te, p, average="binary" if binary else "macro", zero_division=0),
                 "recall": M.recall_score(y_te, p, average="binary" if binary else "macro", zero_division=0),
                 "f1": M.f1_score(y_te, p, average="binary" if binary else "macro", zero_division=0)}
            if binary and hasattr(pipe, "predict_proba"):
                r["roc_auc"] = M.roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
        else:
            r = {"model": name, "MAE": M.mean_absolute_error(y_te, p),
                 "RMSE": np.sqrt(M.mean_squared_error(y_te, p)), "R2": M.r2_score(y_te, p)}
        rows.append(r)
    key = ("roc_auc" if binary else "f1") if IS_CLF else "R2"
    results = pd.DataFrame(rows).set_index("model").sort_values(key, ascending=False)
    BEST = results.index[0] if not results.index[0].startswith("Baseline") else results.index[1]
    print("best model:", BEST, "| ranked by", key)
    display(results)

In [ ]:
# Confusion matrix / actual-vs-predicted for the best model
if results is not None:
    best = fitted[BEST]; p = best.predict(X_te)
    if IS_CLF:
        M.ConfusionMatrixDisplay.from_predictions(y_te, p, cmap="Blues", colorbar=False)
        plt.title(f"{BEST}: confusion matrix"); save_chart("confusion_matrix"); plt.show()
        print(M.classification_report(y_te, p, zero_division=0))
    else:
        plt.scatter(y_te, p, alpha=.4, color=ACCENT); lim = [min(y_te.min(), p.min()), max(y_te.max(), p.max())]
        plt.plot(lim, lim, "k--", lw=1); plt.xlabel("actual"); plt.ylabel("predicted"); plt.title(f"{BEST}: actual vs predicted")
        save_chart("actual_vs_predicted"); plt.show()

## 6. Drivers — what actually matters

Permutation importance = *"shuffle this column and see how much worse the model gets"*.
Works on the original column names, so it's slide-ready.

In [ ]:
from sklearn.inspection import permutation_importance
if results is not None:
    scoring = ("roc_auc" if binary else "f1_macro") if IS_CLF else "r2"
    pi = permutation_importance(fitted[BEST], X_te, y_te, n_repeats=5, random_state=SEED, scoring=scoring, n_jobs=-1)
    imp = pd.Series(pi.importances_mean, index=X_te.columns).sort_values(ascending=False)
    ax = imp.head(10).iloc[::-1].plot.barh(color=ACCENT)
    ax.set_title(f"Top drivers of {TARGET}"); ax.set_xlabel(f"drop in {scoring} when shuffled")
    save_chart("top_drivers"); plt.show()
    display(imp.head(15).to_frame("importance"))
    # Direction check for the top numeric drivers: does more of X mean more of the target?
    for c in [c for c in imp.head(5).index if c in feats_num]:
        q = pd.qcut(data[c].rank(method="first"), 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
        print(f"{c}: target mean by quartile ->", data.groupby(q)[TARGET].mean().round(3).to_dict() if is_numeric_dtype(data[TARGET]) else "")

## 7. Lift / business impact (binary targets)

Turns the model into a decision: *"If we only contact the riskiest 20% of customers, we reach X% of everyone who would churn."*

In [ ]:
if results is not None and IS_CLF and binary:
    proba = fitted[BEST].predict_proba(X_te)[:, 1]
    t = pd.DataFrame({"y": np.asarray(y_te), "p": proba}).sort_values("p", ascending=False).reset_index(drop=True)
    t["decile"] = (np.arange(len(t)) * 10 // len(t)) + 1
    lift = t.groupby("decile").agg(n=("y", "size"), positives=("y", "sum"), rate=("y", "mean"))
    lift["cum_%_of_positives_caught"] = lift["positives"].cumsum() / lift["positives"].sum() * 100
    lift["lift_vs_random"] = lift["rate"] / t["y"].mean()
    display(lift)
    x = np.arange(0, 11) * 10; ycap = np.r_[0, lift["cum_%_of_positives_caught"].values]
    plt.plot(x, ycap, marker="o", lw=2, color=ACCENT, label="model"); plt.plot([0, 100], [0, 100], "--", c="grey", label="random")
    plt.xlabel("% of population targeted (highest risk first)"); plt.ylabel("% of positives caught"); plt.legend()
    plt.title("Gains chart"); save_chart("gains_chart"); plt.show()
    print(f"HEADLINE: targeting the top 20% catches {lift['cum_%_of_positives_caught'].iloc[1]:.0f}% of positives "
          f"({lift['lift_vs_random'].iloc[:2].mean():.1f}x better than random)")

## 8. Segments (optional) — groups of similar rows, each with a profile

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
RUN_SEGMENTS = False      # flip to True if the problem is about customer types / personas
SEG_COLS = []             # e.g. ["recency_days", "frequency", "monetary"]; empty = all numeric non-target

if RUN_SEGMENTS:
    cols = SEG_COLS or [c for c in num_cols if c not in [TARGET] + ID_COLS][:10]
    Z = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(df[cols]))
    sample = np.random.RandomState(SEED).choice(len(Z), min(len(Z), 5000), replace=False)
    scores = {k: silhouette_score(Z[sample], KMeans(k, n_init=10, random_state=SEED).fit_predict(Z[sample])) for k in range(2, 7)}
    k = max(scores, key=scores.get); print("silhouette by k:", {a: round(b, 3) for a, b in scores.items()}, "-> k =", k)
    df["segment"] = KMeans(k, n_init=10, random_state=SEED).fit_predict(Z)
    prof = df.groupby("segment")[cols + ([TARGET] if TARGET in num_cols else [])].mean()
    prof.insert(0, "size_%", df["segment"].value_counts(normalize=True).sort_index() * 100)
    display(prof.style.background_gradient(axis=0, cmap="Blues"))
    print("Name each segment in plain words (e.g. 'big spenders who stopped coming') — that's the slide.")

## 9. Slide numbers — copy these into the deck

Fill this in as you go. One line per slide.

In [ ]:
if results is not None:
    r = results.loc[BEST]; b = results.iloc[[i for i, n in enumerate(results.index) if n.startswith("Baseline")][0]]
    k = ("roc_auc" if binary else "f1") if IS_CLF else "R2"
    print(f"Model: {BEST} | {k} = {r[k]:.3f} vs baseline {b[k]:.3f}")
    print("Top 3 drivers:", ", ".join(imp.head(3).index))
print("Charts saved:", sorted(os.listdir("charts")))